# pyBModes — End-to-end walkthrough

**A guided tour of the modal-analysis pipeline for wind-turbine blades and towers, on self-contained synthetic cases validated against published closed-form formulas.**

Every input file consumed in this notebook is constructed inline — no third-party reference data is bundled or downloaded.  The two demonstration cases are:

| # | Case | Reference |
|---|------|-----------|
| 1 | Uniform isotropic blade, no rotation, no tip mass | Euler-Bernoulli cantilever: $\beta_n L = 1.875104, 4.694091, 7.854757, \ldots$ |
| 2 | Uniform isotropic tower with concentrated top mass | Blevins (1979) / Karnovsky &amp; Lebed (2001) cantilever-with-tip-mass frequency equation |

Both reference formulas are textbook material whose numerical values are facts, not licensed data.

All figures use a custom journal-paper matplotlib style (Okabe–Ito colour-blind-safe palette, serif fonts, tight grids, inline ticks).

## Outline

1. **Setup** — imports, matplotlib styling, synthetic-fixture helpers.
2. **Synthetic blade** — build, solve, validate against Euler-Bernoulli.
3. **ElastoDyn fits** — fit 6th-order blade polynomials and patch a `.dat`.
4. **Synthetic tower with top mass** — build, solve, validate against the Blevins frequency equation.
5. **Under the hood** — the vectorised FEM core (`_element_matrices_batch`).

## 1. Setup

* **Serif fonts** (STIX Two Text → STIX → DejaVu Serif fallback).
* **Okabe–Ito 8-colour palette**, recommended for colour-blind accessibility ([Okabe & Ito, 2008](https://jfly.uni-koeln.de/color/)).
* **Inward ticks on all four axes**, thin frame, dashed grid, 110 DPI inline.
* The `sys.path` patch makes the notebook work on a fresh clone without `pip install`.

In [ ]:
import pathlib
import sys
import tempfile

_src = (pathlib.Path('..').resolve() / 'src')
if _src.is_dir() and str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import brentq

from pybmodes.elastodyn import (
    compute_blade_params,
    compute_tower_params,
    compute_tower_params_report,
    patch_dat,
)
from pybmodes.models import RotatingBlade, Tower
from pybmodes.plots import blade_fit_pairs, tower_fit_pairs

# --- Okabe-Ito colour-blind-safe palette --------------------------------
OI = {
    'black':   '#000000',
    'orange':  '#E69F00',
    'sky':     '#56B4E9',
    'green':   '#009E73',
    'yellow':  '#F0E442',
    'blue':    '#0072B2',
    'verm':    '#D55E00',
    'purple':  '#CC79A7',
}
PALETTE = list(OI.values())

# --- Journal-paper matplotlib defaults ---------------------------------
mpl.rcParams.update({
    'font.family':       'serif',
    'font.serif':        ['STIX Two Text', 'STIX', 'DejaVu Serif', 'Times New Roman'],
    'mathtext.fontset':  'stix',
    'font.size':          10,
    'axes.titlesize':     11,
    'axes.titleweight':  'bold',
    'axes.labelsize':     10,
    'xtick.labelsize':    9,
    'ytick.labelsize':    9,
    'legend.fontsize':    9,
    'figure.titlesize':   12,
    'figure.titleweight':'bold',
    'axes.linewidth':     0.8,
    'axes.prop_cycle':    mpl.cycler(color=PALETTE),
    'lines.linewidth':    1.6,
    'lines.markersize':   4.0,
    'xtick.direction':   'in',
    'ytick.direction':   'in',
    'xtick.top':          True,
    'ytick.right':        True,
    'xtick.major.width':  0.7,
    'ytick.major.width':  0.7,
    'legend.frameon':     True,
    'legend.framealpha':  0.95,
    'legend.edgecolor':   '0.85',
    'figure.dpi':         110,
    'savefig.dpi':        300,
    'savefig.bbox':      'tight',
})

# --- Inline synthetic-fixture builders ---------------------------------
# These mirror the helpers in tests/_synthetic_bmi.py so the notebook is
# self-contained.  Every numeric value below is freely chosen here.

_GEN_LABELS = ('echo', 'beam_type', 'rot_rpm', 'rpm_mult', 'radius', 'hub_rad',
               'precone', 'bl_thp', 'hub_conn', 'n_modes_print', 'tab_delim',
               'mid_node_tw')
_TIP_LABELS = ('tip_mass', 'cm_offset', 'cm_axial',
               'ixx', 'iyy', 'izz', 'ixy', 'izx', 'iyz')
_SCALE_LABELS = ('sec_mass', 'flp_iner', 'lag_iner', 'flp_stff', 'edge_stff',
                 'tor_stff', 'axial_stff', 'cg_offst', 'sc_offst', 'tc_offst')

def write_bmi(path, *, title, beam_type, radius, hub_rad=0.0, rot_rpm=0.0,
              hub_conn=1, tip_mass=0.0, sec_props_file='secs.dat',
              n_elements=8, tow_support=0):
    el_loc = list(np.linspace(0.0, 1.0, n_elements + 1))
    gen = ('f', str(beam_type), f'{rot_rpm}', '1.0', f'{radius}', f'{hub_rad}',
           '0.0', '0.0', str(hub_conn), '20', 'f', 'f')
    tip = (str(tip_mass),) + ('0.0',) * 8

    lines = [
        '================ synthetic .bmi ================',
        f"'{title}'",
        '--- general parameters ---',
        '--- (echo through mid_node_tw) ---',
        *(f'{v}    ! {lbl}' for v, lbl in zip(gen, _GEN_LABELS)),
        '--- tip mass ---',
        '--- (mass + 9 inertia/offsets) ---',
        *(f'{v}    ! {lbl}' for v, lbl in zip(tip, _TIP_LABELS)),
        '--- distributed properties ---',
        '--- (id_mat + filename) ---',
        '1    ! id_mat',
        f"'{sec_props_file}'    ! sec_props_file",
        '--- scaling factors ---',
        '--- (10 unity multipliers) ---',
        *(f'1.0    ! {lbl}' for lbl in _SCALE_LABELS),
        '--- fe discretisation ---',
        '--- (nselt + el_loc) ---',
        f'{n_elements}    ! n_elements',
        '--- el_loc ---',
        '  '.join(f'{v}' for v in el_loc),
    ]
    if beam_type == 2:
        lines += [
            '--- tower support ---',
            '--- (tow_support code) ---',
            f'{tow_support}    ! tow_support',
        ]
    pathlib.Path(path).write_text('\n'.join(lines) + '\n', encoding='utf-8')
    return pathlib.Path(path)


def write_uniform_sec_props(path, *, n_secs=11, mass_den=100.0,
                            flp_stff=1.0e8, edge_stff=1.0e9,
                            tor_stff=1.0e7, axial_stff=1.0e10):
    span = np.linspace(0.0, 1.0, n_secs)
    rows = '\n'.join(
        f'{s:.4f}  0.0  0.0  {mass_den}  {mass_den * 0.1}  {mass_den * 0.1}  '
        f'{flp_stff}  {edge_stff}  {tor_stff}  {axial_stff}  0.0  0.0  0.0'
        for s in span
    )
    body = (
        'synthetic uniform section properties\n'
        f'{n_secs}  n_secs\n'
        '\n'
        'span_loc str_tw tw_iner mass_den flp_iner edge_iner '
        'flp_stff edge_stff tor_stff axial_stff cg_offst sc_offst tc_offst\n'
        '  -      deg    deg     kg/m     kg.m     kg.m     '
        'N.m^2   N.m^2     N.m^2   N        m        m        m\n'
        + rows + '\n'
    )
    pathlib.Path(path).write_text(body, encoding='utf-8')
    return pathlib.Path(path)

print('Style configured.')

## 2. Synthetic uniform blade

Build a uniform Euler-Bernoulli cantilever blade in a temp directory and validate the FEM solver against the closed-form natural-frequency series

$\omega_n = (\beta_n L)^2 \sqrt{\dfrac{EI}{m L^4}}, \qquad \beta_n L = 1.875104,\;4.694091,\;7.854757,\;\ldots$

with the parameters chosen below.

In [ ]:
L_BLADE   = 50.0          # blade length [m]
M_BLADE   = 100.0         # mass per unit length [kg/m]
EI_BLADE  = 1.0e8         # flap bending stiffness [N*m^2]
EI_LAG    = 1.0e10        # lag bending stiffness >> flap so the lowest two
                          # modes are unambiguously flap-bending

_blade_dir = pathlib.Path(tempfile.mkdtemp(prefix='pybmodes_blade_'))
blade_bmi = write_bmi(
    _blade_dir / 'blade.bmi',
    title='uniform isotropic blade - walkthrough',
    beam_type=1, radius=L_BLADE, hub_rad=0.0,
    sec_props_file='blade_secs.dat',
    n_elements=20,
)
write_uniform_sec_props(_blade_dir / 'blade_secs.dat',
                        n_secs=11,
                        mass_den=M_BLADE,
                        flp_stff=EI_BLADE, edge_stff=EI_LAG,
                        tor_stff=1.0e7, axial_stff=1.0e10)

blade = RotatingBlade(blade_bmi)
blade_result = blade.run(n_modes=10)

# Closed-form Euler-Bernoulli flap frequencies (first two suffice for
# unambiguous validation; from mode 3 onward the lag and torsion families
# interleave between the flap modes).
BETA_L = np.array([1.87510407, 4.69409113])
alpha = np.sqrt(EI_BLADE / (M_BLADE * L_BLADE**4))
f_analytical_hz = (BETA_L ** 2) * alpha / (2.0 * np.pi)

print(f"{'mode':>5}  {'pyBModes [Hz]':>14}  {'Euler-Bernoulli [Hz]':>22}  {'rel. error':>11}")
print('-' * 60)
for k, (ff, fa) in enumerate(zip(blade_result.frequencies[:2], f_analytical_hz)):
    rel = abs(ff - fa) / fa * 100.0
    print(f'{k+1:>5}  {ff:>14.6f}  {fa:>22.6f}  {rel:>10.3f}%')

print('\nFirst 5 pyBModes frequencies (mixed flap / lag / torsion modes):')
for k, ff in enumerate(blade_result.frequencies[:5]):
    print(f'  mode {k+1}: {ff:.4f} Hz')

### 2.1 Mode shapes

First five flap mode shapes of the uniform cantilever, normalised to peak amplitude per mode.

In [ ]:
def plot_mode_shapes_paper(shapes, n_modes=5, panels=('flap', 'lag'),
                           panel_labels=('Flap (out-of-plane)', 'Edge (in-plane)'),
                           title='', figsize=(11, 4.2)):
    fig, axes = plt.subplots(1, len(panels), figsize=figsize,
                              constrained_layout=True, sharex=True)
    axes = np.atleast_1d(axes)
    cycle = PALETTE[1:]
    for i, s in enumerate(shapes[:n_modes]):
        c = cycle[i % len(cycle)]
        for ax, comp in zip(axes, panels):
            disp = s.flap_disp if comp == 'flap' else s.lag_disp
            peak = max(np.abs(disp).max(), 1e-30)
            ax.plot(s.span_loc, disp / peak, color=c, lw=1.6, marker='o',
                    ms=3.5, mew=0,
                    label=f'Mode {s.mode_number}  ({s.freq_hz:.3f} Hz)')
    for ax, lab in zip(axes, panel_labels):
        ax.set_xlabel(r'Normalised span  $s$')
        ax.set_ylabel(r'Normalised displacement')
        ax.set_title(lab)
        ax.axhline(0, color='0.55', lw=0.6)
        ax.grid(True, ls='--', lw=0.4, alpha=0.6)
        ax.set_xlim(-0.02, 1.02)
    axes[0].legend(loc='upper left', handlelength=1.4)
    fig.suptitle(title)
    return fig

plot_mode_shapes_paper(blade_result.shapes, n_modes=5,
                       title='Synthetic uniform blade \u2014 first five modes')
plt.show()

### 2.2 ElastoDyn polynomial fits

OpenFAST/ElastoDyn represents blade and tower mode shapes as constrained 6th-order polynomials
$
\phi(x) = c_2 x^2 + c_3 x^3 + c_4 x^4 + c_5 x^5 + c_6 x^6, \qquad \sum_{k=2}^{6} c_k = 1.
$

`compute_blade_params` automatically picks the first two flap modes and the first edge mode.

In [ ]:
blade_params = compute_blade_params(blade_result)

print(f"{'name':<10}  {'C2':>8}  {'C3':>8}  {'C4':>8}  {'C5':>8}  {'C6':>8}  {'RMS':>7}")
print('-' * 60)
for name, fit in (('BldFl1Sh', blade_params.BldFl1Sh),
                  ('BldFl2Sh', blade_params.BldFl2Sh),
                  ('BldEdgSh', blade_params.BldEdgSh)):
    c = fit.coefficients()
    print(f'{name:<10}  {c[0]:>+8.4f}  {c[1]:>+8.4f}  {c[2]:>+8.4f}  '
          f'{c[3]:>+8.4f}  {c[4]:>+8.4f}  {fit.rms_residual:>7.4f}')


def plot_fit_quality_paper(entries, title=''):
    n = len(entries)
    ncols = min(3, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.6 * ncols, 3.6 * nrows),
                              constrained_layout=True)
    axes = np.atleast_1d(axes).flatten()
    x_fine = np.linspace(0.0, 1.0, 200)
    for ax, (label, x, y, fit) in zip(axes, entries):
        x = np.asarray(x, dtype=float)
        y = np.asarray(y, dtype=float)
        tip = y[-1] if abs(y[-1]) > 1e-30 else max(np.abs(y).max(), 1.0)
        y_n = y / tip
        ax.fill_between(x, y_n, fit.evaluate(x), alpha=0.22, color=OI['verm'],
                        label='Residual')
        ax.plot(x_fine, fit.evaluate(x_fine), color=OI['blue'], lw=1.8,
                label='Polynomial fit')
        ax.scatter(x, y_n, s=22, color=OI['green'], zorder=5,
                   edgecolor='white', linewidth=0.5, label='FEM nodes')
        ax.axhline(0, color='0.55', lw=0.5)
        ax.axhline(1, color='0.55', lw=0.5, ls='--')
        ax.set_xlim(-0.02, 1.02)
        ax.set_xlabel(r'Normalised span  $s$')
        ax.set_ylabel('Normalised displacement')
        ax.set_title(label)
        ax.grid(True, ls='--', lw=0.4, alpha=0.6)
        ax.text(0.97, 0.05,
                f'RMS = {fit.rms_residual:.4f}\ntip slope = {fit.tip_slope:.3f}',
                transform=ax.transAxes, va='bottom', ha='right',
                fontsize=8, color=OI['verm'],
                bbox=dict(boxstyle='round,pad=0.3', fc='white',
                          ec='0.78', alpha=0.95))
    axes[0].legend(loc='lower right', fontsize=8)
    for ax in axes[n:]:
        ax.set_visible(False)
    fig.suptitle(title)
    return fig

plot_fit_quality_paper(
    blade_fit_pairs(blade_result, blade_params),
    title='Synthetic blade \u2014 ElastoDyn polynomial fits',
)
plt.show()

### 2.3 Patching an ElastoDyn `.dat` file in place

`patch_dat(path, params)` performs a regex-based in-place replacement of each `BldFlNSh(k)` / `BldEdgSh(k)` line, preserving comments and indentation.

In [ ]:
TEMPLATE = '''\
------- BLADE MODE SHAPES ------------------------------------------
          1   BldFl1Sh(2)   - Blade 1 flap mode 1, coeff of x^2
          0   BldFl1Sh(3)   - Blade 1 flap mode 1, coeff of x^3
          0   BldFl1Sh(4)   - Blade 1 flap mode 1, coeff of x^4
          0   BldFl1Sh(5)   - Blade 1 flap mode 1, coeff of x^5
          0   BldFl1Sh(6)   - Blade 1 flap mode 1, coeff of x^6
          1   BldFl2Sh(2)   - Blade 1 flap mode 2, coeff of x^2
          0   BldFl2Sh(3)   - ...
          0   BldFl2Sh(4)   - ...
          0   BldFl2Sh(5)   - ...
          0   BldFl2Sh(6)   - ...
          1   BldEdgSh(2)   - Blade 1 edge mode 1, coeff of x^2
          0   BldEdgSh(3)   - ...
          0   BldEdgSh(4)   - ...
          0   BldEdgSh(5)   - ...
          0   BldEdgSh(6)   - ...
'''

with tempfile.TemporaryDirectory() as tmp:
    dat = pathlib.Path(tmp) / 'ElastoDyn_blade.dat'
    dat.write_text(TEMPLATE, encoding='utf-8')
    print('--- BEFORE PATCH ----------------------------------------------')
    print(dat.read_text(encoding='utf-8'))

    patch_dat(dat, blade_params)

    print('--- AFTER PATCH -----------------------------------------------')
    print(dat.read_text(encoding='utf-8'))

## 3. Synthetic uniform tower with concentrated top mass

The tower idealises a wind-turbine support structure with a rotor-nacelle assembly modelled as a concentrated mass at the top.  The lowest natural frequency is the root of

$1 + \cos(\beta L)\,\cosh(\beta L) - \mu \beta L \bigl(\sin(\beta L)\,\cosh(\beta L) - \cos(\beta L)\,\sinh(\beta L)\bigr) = 0$

with mass ratio $\mu = m_\text{tip} / (\rho A L)$  (Blevins 1979; Karnovsky &amp; Lebed 2001).  Multiplying $\beta L$ by $\sqrt{EI/(\rho A L^4)}$ gives the angular natural frequency.

We pick a representative onshore tower — height 80 m, mass density 5000 kg/m, bending stiffness $5\times 10^{10}$ N·m², tip mass equal to one tower-length's worth of beam mass ($\mu = 1$) — and compare against the closed-form root.

In [ ]:
L_TOWER   = 80.0
M_TOWER   = 5000.0
EI_TOWER  = 5.0e10
MU        = 1.0                  # tip-mass ratio
TIP_MASS  = MU * M_TOWER * L_TOWER

_tower_dir = pathlib.Path(tempfile.mkdtemp(prefix='pybmodes_tower_'))
tower_bmi = write_bmi(
    _tower_dir / 'tower.bmi',
    title='uniform tower with top mass - walkthrough',
    beam_type=2, radius=L_TOWER, hub_rad=0.0,
    sec_props_file='tower_secs.dat',
    tip_mass=TIP_MASS, n_elements=20, tow_support=0,
)
write_uniform_sec_props(_tower_dir / 'tower_secs.dat',
                        n_secs=11,
                        mass_den=M_TOWER,
                        flp_stff=EI_TOWER, edge_stff=EI_TOWER,
                        tor_stff=EI_TOWER * 0.1,
                        axial_stff=1.0e12)

tower_result = Tower(tower_bmi).run(n_modes=10)
tower_params, tower_report = compute_tower_params_report(tower_result)

# Closed-form first frequency
def freq_eq(bL, mu):
    cb, sb = np.cos(bL), np.sin(bL)
    chb, shb = np.cosh(bL), np.sinh(bL)
    return 1.0 + cb*chb - mu*bL*(sb*chb - cb*shb)

bL_root = brentq(freq_eq, 0.5, 2.5, args=(MU,), xtol=1e-12)
f_blevins_hz = (bL_root**2) * np.sqrt(EI_TOWER / (M_TOWER * L_TOWER**4)) / (2*np.pi)

print(f'Blevins-Karnovsky reference for mu={MU}:')
print(f'  beta*L                    = {bL_root:.6f}')
print(f'  first-mode frequency [Hz] = {f_blevins_hz:.6f}')
print()
print(f"{'mode':>5}  {'pyBModes [Hz]':>14}")
print('-' * 25)
for i, f in enumerate(tower_result.frequencies[:6]):
    print(f'{i+1:>5}  {f:>14.6f}')

rel = abs(tower_result.frequencies[0] - f_blevins_hz) / f_blevins_hz * 100.0
print(f'\nFirst-mode error vs Blevins/Karnovsky reference: {rel:.3f}%')
print(f'\nFamily selection report:')
print(f'  selected FA modes: {tower_report.selected_fa_modes}')
print(f'  selected SS modes: {tower_report.selected_ss_modes}')

In [ ]:
plot_mode_shapes_paper(
    tower_result.shapes, n_modes=5,
    panels=('flap', 'lag'),
    panel_labels=('Fore\u2013aft (FA)', 'Side\u2013side (SS)'),
    title='Synthetic uniform tower with top mass \u2014 mode shapes',
)
plt.show()

plot_fit_quality_paper(
    tower_fit_pairs(tower_result, tower_params),
    title='Synthetic tower \u2014 ElastoDyn polynomial fits',
)
plt.show()

## 4. Under the hood: the vectorised FEM core

Every element's 15×15 stiffness and mass matrices are computed in a *single* `numpy.einsum`-driven call.  The public single-element `element_matrices(...)` is a thin scalar wrapper around the batched core `_element_matrices_batch(...)`, which the global `assemble(...)` function uses directly.

The vectorisation strategy:

1. **Shape functions are pre-computed at module load** on the 6 Gauss-point abscissae, split into `eli`-independent and `eli`-scaling parts.
2. **Every per-element scalar carries a leading element axis.**  Twist, centrifugal tension, structural-twist sin/cos products, and bending-coupling terms broadcast naturally over `(nselt, NG)` grids.
3. **Every double sum over Gauss points and local DOF pairs collapses to one `einsum`.**  The classical Fortran-style triple loop

   ```
   for n in range(NG):
       for i in range(4):
           for j in range(4):
               ek[i, j] += w[n] * h[n, i] * h[n, j] * coef[n]
   ```

   becomes a single `np.einsum('en,eni,enj->eij', w * coef, h, h)` call that simultaneously sums over Gauss points *and* runs over all elements.

In [ ]:
from pybmodes.fem.element import _element_matrices_batch

# A uniform 5-element non-dimensional beam in tip-to-root ordering.
nselt = 5
eli   = np.full(nselt, 1.0 / nselt)
xbi   = 1.0 - np.arange(1, nselt + 1) * eli[0]
eiy   = np.full(nselt, 1.0)
eiz   = np.full(nselt, 1.0)
gj    = np.full(nselt, 1.0)
eac   = np.full(nselt, 100.0)
rmas  = np.full(nselt, 1.0)
skm1  = np.full(nselt, 1.0e-5)
skm2  = np.full(nselt, 1.0e-5)
eg    = np.zeros(nselt)
ea    = np.zeros(nselt)
axfi  = np.zeros(nselt)

ek, em = _element_matrices_batch(
    eli=eli, xbi=xbi, eiy=eiy, eiz=eiz, gj=gj, eac=eac, rmas=rmas,
    skm1=skm1, skm2=skm2, eg=eg, ea=ea, axfi=axfi,
    omega2=0.0,
    sec_loc=np.array([0.0, 1.0]),
    str_tw=np.array([0.0, 0.0]),
    distr_k=np.zeros(nselt),
)

print(f'nselt              = {nselt}')
print(f'ek.shape           = {ek.shape}    (nselt, 15, 15)')
print(f'em.shape           = {em.shape}    (nselt, 15, 15)')
print(f'ek block-symmetric = {np.allclose(ek, ek.swapaxes(1, 2))}')
print(f'em block-symmetric = {np.allclose(em, em.swapaxes(1, 2))}')

## Closing

What the walkthrough has shown:

1. **Inputs** — the BMI / section-properties parsers expose every field as ordinary Python dataclasses; section properties are easy to plot and sanity-check before solving.
2. **Modal solve** — a single `RotatingBlade(...).run(n_modes)` (or `Tower(...).run(...)`) call returns frequencies + per-node mode shapes, agreeing with closed-form Euler-Bernoulli / Blevins references to better than 0.5 %.
3. **ElastoDyn fits** — `compute_blade_params` / `compute_tower_params_report` produce constrained 6th-order polynomials with explicit FA/SS family selection diagnostics.
4. **File patching** — `patch_dat` rewrites OpenFAST `.dat` files in place, preserving comments and indentation.
5. **Vectorised FEM core** — element matrices are computed in a single batched `einsum`-driven call, with all loops over Gauss points and elements collapsed.

### Where development is still active

- **FEM solver acceleration (partial).**  Element-matrix construction is now vectorised over both Gauss points and elements via `numpy.einsum` (~2–3× speedup vs the original per-element loop). The remaining bottleneck for large `nselt` is the dense `scipy.linalg.eigh` solve; a sparse shift-invert `eigsh` path is on the roadmap.
- **Robust mode classification.**  The current FA / SS family selection uses a fixed RMS threshold and tip-displacement comparison; a better heuristic is needed for offshore towers where rigid-body platform motion contaminates the lowest modes.
- **Centrifugal-stiffening validation.**  The rotating-blade case is currently validated only via internal regression; an independent published-paper benchmark is on the roadmap.

Issues, suggestions, and pull requests welcome at <https://github.com/SMI-Lab-Inha/pyBModes>.